## CELL 1: Imports

In [ ]:
# === Cell 1: 強制安裝 CatBoost,並驗證 ===
!pip install -q catboost

# 確保安裝成功 — 這行如果報錯就 stop,不要往下跑
import catboost
print(f"✅ catboost {catboost.__version__} ready")

# 確認 GPU
import torch
assert torch.cuda.is_available()
print(f"✅ {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00
✅ catboost 1.2.10 ready
✅ Tesla T4


In [ ]:
# ============================================================
# AI CUP 2026 桌球預測 — V6 (V5 乾淨版 + Prior-Correction 後處理)
# Public LB 0.3929 | rank ~33/300
# ============================================================
# 架構: Bi-GRU 多任務 NN + LGB/XGB/Cat GBDT ensemble + position masking
# 每個 cell 對應一個功能模組,從上到下依序執行,最後一個 cell 跑 main()
# ============================================================

import os, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from collections import Counter
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    print("⚠️  catboost missing, install with: !pip install -q catboost")
    HAS_CATBOOST = False


# ============================================================

## CELL 2: Config (所有超參數)

In [ ]:
class Config:
    train_path     = "/kaggle/input/datasets/kgiprkoio/table-tennis/train.csv"
    test_path      = "/kaggle/input/datasets/kgiprkoio/table-tennis/test_new.csv"
    sample_path    = "/kaggle/input/datasets/kgiprkoio/table-tennis/sample_submission.csv"
    old_test_path  = "/kaggle/input/datasets/kgiprkoio/table-tennis/test.csv"
    output_path    = "/kaggle/working/submission_final_v6.csv"

    # --- 5 fold ---
    seed = 42
    n_folds = 5

    # --- NN ---
    n_seeds_nn = 2                  # NN seeds per fold (same as VV1)
    batch_size = 128
    epochs = 40
    lr = 1e-3
    min_lr = 1e-5
    weight_decay = 1e-4
    patience = 8
    grad_clip = 1.0
    max_seq_len = 64
    d_model = 128
    dropout = 0.2
    rally_end_horizon = 8
    use_transformer = True            # 加 Transformer 當 ensemble 第三票
    use_chain         = False         # chain 實測拖累 OOF, 關閉
    fix_server_leak   = True          # 擋掉 test.csv 的 serverGetPoint 標籤(防洩漏)
    use_shuttlenet    = True          # 加 ShuttleNet 啟發版當 ensemble 成員

    w_action    = 0.40
    w_point     = 0.40
    w_server    = 0.05
    w_rally_end = 0.40

    use_class_weight = True
    label_smoothing  = 0.05
    player_dropout_p = 0.30

    te_smoothing_alpha = 20.0

    # --- GBDT ---
    n_seeds_gbdt = 3                # VV2A: multi-seed bagging
    n_estimators = 400
    use_xgb = True
    use_cat = True
    use_rf_et_action = True   # 加 RandomForest+ExtraTrees 到 action GBDT blend (本機 unseen-OOF 驗證 +~0.003)
    use_lstm          = True         # opt-in: 加 Bi-LSTM 成員 (約 +1h, 權搜自動定權重)
    use_tcn           = True         # opt-in: 加 causal TCN 成員 (約 +1h, 同上)

    gbdt_action_weight = 0.70
    gbdt_point_weight  = 0.70
    gbdt_server_weight = 0.30

    lgb_weight_in_gbdt = 0.40
    xgb_weight_in_gbdt = 0.30
    cat_weight_in_gbdt = 0.30

    # --- Ensemble Weight Search (VV3: expanded range) ---
    do_weight_search = True
    weight_search_radius = 0.20     # was 0.10 in VV1; ±0.20 to reach high-GBDT regions
    weight_search_step = 0.05
    weight_search_overfit_threshold = 0.015


CFG = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"VV3: device={device}, CatBoost={HAS_CATBOOST}")
print(f"  n_seeds_nn={CFG.n_seeds_nn}, n_seeds_gbdt={CFG.n_seeds_gbdt}, "
      f"weight_radius=±{CFG.weight_search_radius}")
USE_AMP = (device.type == "cuda")
if not HAS_CATBOOST: CFG.use_cat = False



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (device.type == 'cuda')
print(f'device={device}, CatBoost={HAS_CATBOOST}')
if not HAS_CATBOOST: CFG.use_cat = False

VV3: device=cuda, CatBoost=True
  n_seeds_nn=3, n_seeds_gbdt=3, weight_radius=±0.2
device=cuda, CatBoost=True


## CELL 3: 常數 + seed + 欄位定義

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


N_ACTION = 19
N_POINT  = 10
PAD_IDX  = 0
UNK_IDX  = 1

ATTACK_ACTIONS    = {1, 2, 3, 4, 5, 6, 7}
CONTROL_ACTIONS   = {8, 9, 10, 11}
DEFENSIVE_ACTIONS = {12, 13, 14}

CAT_COLS_NORMAL = ["sex", "numberGame", "strikeId", "handId",
                   "strengthId", "spinId", "positionId"]
PLAYER_COLS = ["gamePlayerId", "gamePlayerOtherId"]
NUM_COLS_BASE    = ["scoreSelf", "scoreOther", "strikeNumber"]
NUM_COLS_DERIVED = ["scoreDiff", "scoreSum", "strike_parity",
                    "rally_id_norm", "next_is_main"]

PLAYER_TE_COLS = [
    "self_act_top1_p",  "self_pt_top1_p",  "self_act_ent",
    "self_forehand_freq", "self_pos_left_freq", "self_pos_right_freq",
    "other_act_top1_p", "other_pt_top1_p", "other_act_ent",
    "other_forehand_freq", "other_pos_left_freq", "other_pos_right_freq",
    "next_act_top1_p",  "next_pt_top1_p",  "next_act_ent",
    "next_forehand_freq", "next_pos_left_freq", "next_pos_right_freq",
]
NUM_COLS = NUM_COLS_BASE + NUM_COLS_DERIVED + PLAYER_TE_COLS


# ============================================================
# Stats helpers (player TE, sex TE, point conditional, action bigram)
# ============================================================

## CELL 4: 統計 helper: player TE / sex stats

In [ ]:
def _agg_top1(s):
    c = s.value_counts(normalize=True)
    return float(c.iloc[0]) if len(c) else 0.1

def _agg_ent(s):
    c = s.value_counts(normalize=True)
    return float(-(c * np.log(c + 1e-9)).sum()) if len(c) else 2.5


def compute_player_stats(train_raw, alpha=None):
    if alpha is None: alpha = CFG.te_smoothing_alpha
    actor = np.where(train_raw["strikeNumber"].values % 2 == 1,
                     train_raw["gamePlayerId"].values, train_raw["gamePlayerOtherId"].values)
    df = pd.DataFrame({"actor": actor, "actionId": train_raw["actionId"].values,
        "pointId": train_raw["pointId"].values, "handId": train_raw["handId"].values,
        "positionId": train_raw["positionId"].values})
    g = df.groupby("actor")
    raw = pd.DataFrame({
        "act_top1_p": g["actionId"].agg(_agg_top1), "pt_top1_p": g["pointId"].agg(_agg_top1),
        "act_ent": g["actionId"].agg(_agg_ent),
        "forehand_freq": g["handId"].agg(lambda x: float((x == 1).mean())),
        "pos_left_freq": g["positionId"].agg(lambda x: float((x == 1).mean())),
        "pos_right_freq": g["positionId"].agg(lambda x: float((x == 3).mean())),
    })
    counts = g.size().rename("count")
    global_priors = {"act_top1_p": _agg_top1(df["actionId"]), "pt_top1_p": _agg_top1(df["pointId"]),
        "act_ent": _agg_ent(df["actionId"]),
        "forehand_freq": float((df["handId"] == 1).mean()),
        "pos_left_freq": float((df["positionId"] == 1).mean()),
        "pos_right_freq": float((df["positionId"] == 3).mean())}
    raw = raw.join(counts, how="left")
    for field, prior in global_priors.items():
        raw[field] = (raw["count"] * raw[field] + alpha * prior) / (raw["count"] + alpha)
    return raw.drop(columns=["count"]).reset_index()


def compute_sex_stats(train_raw):
    actor = np.where(train_raw["strikeNumber"].values % 2 == 1,
                     train_raw["gamePlayerId"].values, train_raw["gamePlayerOtherId"].values)
    df = pd.DataFrame({"sex": train_raw["sex"].values, "actionId": train_raw["actionId"].values,
        "pointId": train_raw["pointId"].values, "handId": train_raw["handId"].values,
        "positionId": train_raw["positionId"].values})
    g = df.groupby("sex")
    return pd.DataFrame({
        "act_top1_p": g["actionId"].agg(_agg_top1), "pt_top1_p": g["pointId"].agg(_agg_top1),
        "act_ent": g["actionId"].agg(_agg_ent),
        "forehand_freq": g["handId"].agg(lambda x: float((x == 1).mean())),
        "pos_left_freq": g["positionId"].agg(lambda x: float((x == 1).mean())),
        "pos_right_freq": g["positionId"].agg(lambda x: float((x == 3).mean())),
    }).reset_index()

## CELL 5: 統計 helper: 條件落點 + action bigram

In [ ]:
def compute_point_conditional_stats(train_raw, min_count=20):
    sorted_df = train_raw.sort_values(["rally_uid", "strikeNumber"]).reset_index(drop=True)
    pairs = []
    for uid, g in sorted_df.groupby("rally_uid", sort=False):
        if len(g) < 2: continue
        pos_arr = g["positionId"].values; act_arr = g["actionId"].values; pt_arr = g["pointId"].values
        for k in range(len(g) - 1):
            pairs.append((pos_arr[k], act_arr[k], pt_arr[k + 1]))
    pairs_df = pd.DataFrame(pairs, columns=["pos", "act", "next_pt"])
    cond_stats = {}
    for (p, a), grp in pairs_df.groupby(["pos", "act"]):
        c = grp["next_pt"].value_counts(normalize=True)
        cond_stats[(int(p), int(a))] = {"top1_p": float(c.iloc[0]), "top1_id": int(c.index[0]),
                                          "ent": float(-(c * np.log(c + 1e-9)).sum()), "count": int(len(grp))}
    act_stats = {}
    for a, grp in pairs_df.groupby("act"):
        c = grp["next_pt"].value_counts(normalize=True)
        act_stats[int(a)] = {"top1_p": float(c.iloc[0]), "top1_id": int(c.index[0]),
                              "ent": float(-(c * np.log(c + 1e-9)).sum()), "count": int(len(grp))}
    c_global = pairs_df["next_pt"].value_counts(normalize=True)
    global_stats = {"top1_p": float(c_global.iloc[0]), "top1_id": int(c_global.index[0]),
                     "ent": float(-(c_global * np.log(c_global + 1e-9)).sum()), "count": int(len(pairs_df))}
    print(f"  conditional pt: {len(cond_stats)} (pos, act) pairs")
    return cond_stats, act_stats, global_stats


def lookup_conditional_pt(pos, act, cond_stats, act_stats, global_stats, min_count=20):
    k = (int(pos), int(act))
    if k in cond_stats and cond_stats[k]["count"] >= min_count: return cond_stats[k]
    if int(act) in act_stats: return act_stats[int(act)]
    return global_stats


# VV3 NEW (from VV2A): bigram action statistics (the aggregate, not raw code)
def compute_action_bigram_stats(train_raw, min_count=30):
    sorted_df = train_raw.sort_values(["rally_uid", "strikeNumber"]).reset_index(drop=True)
    triples = []
    for uid, g in sorted_df.groupby("rally_uid", sort=False):
        if len(g) < 3: continue
        a_arr = g["actionId"].values
        for k in range(2, len(g)):
            triples.append((a_arr[k-2], a_arr[k-1], a_arr[k]))
    df = pd.DataFrame(triples, columns=["p2", "p1", "next_a"])
    stats = {}
    for (p2, p1), grp in df.groupby(["p2", "p1"]):
        if len(grp) < min_count: continue
        c = grp["next_a"].value_counts(normalize=True)
        stats[(int(p2), int(p1))] = {"top1_p": float(c.iloc[0]), "top1_id": int(c.index[0]),
                                       "ent": float(-(c * np.log(c + 1e-9)).sum())}
    c_global = df["next_a"].value_counts(normalize=True)
    global_stats = {"top1_p": float(c_global.iloc[0]), "top1_id": int(c_global.index[0]),
                     "ent": float(-(c_global * np.log(c_global + 1e-9)).sum())}
    print(f"  action bigram → next-a: {len(stats)} (p2, p1) pairs with count >= {min_count}")
    return stats, global_stats


def lookup_bigram_a(p2_a, p1_a, bigram_stats, global_stats):
    k = (int(p2_a), int(p1_a))
    if k in bigram_stats: return bigram_stats[k]
    return global_stats


_TE_FIELDS = ["act_top1_p", "pt_top1_p", "act_ent", "forehand_freq", "pos_left_freq", "pos_right_freq"]

## CELL 6: 前處理: attach TE / derived / preprocess

In [ ]:
def attach_player_te(df, player_stats, sex_stats):
    p2stats = {row["actor"]: tuple(row[f] for f in _TE_FIELDS) for _, row in player_stats.iterrows()}
    s2stats = {row["sex"]:   tuple(row[f] for f in _TE_FIELDS) for _, row in sex_stats.iterrows()}
    DEFAULT = (0.30, 0.20, 2.0, 0.5, 0.33, 0.33)
    def _lookup(player_ids, sexes):
        out = np.zeros((len(player_ids), len(_TE_FIELDS)), dtype=np.float32)
        for i, (p, s) in enumerate(zip(player_ids, sexes)):
            if p in p2stats:   out[i] = p2stats[p]
            elif s in s2stats: out[i] = s2stats[s]
            else:              out[i] = DEFAULT
        return out
    self_te  = _lookup(df["gamePlayerId"].values,      df["sex"].values)
    other_te = _lookup(df["gamePlayerOtherId"].values, df["sex"].values)
    next_actor = np.where((df["strikeNumber"].values + 1) % 2 == 1,
                          df["gamePlayerId"].values, df["gamePlayerOtherId"].values)
    next_te = _lookup(next_actor, df["sex"].values)
    for slot, arr in [("self", self_te), ("other", other_te), ("next", next_te)]:
        for j, fname in enumerate(_TE_FIELDS):
            df[f"{slot}_{fname}"] = arr[:, j]
    return df


def add_derived(df):
    df["scoreDiff"]     = df["scoreSelf"] - df["scoreOther"]
    df["scoreSum"]      = df["scoreSelf"] + df["scoreOther"]
    df["strike_parity"] = (df["strikeNumber"] % 2 == 1).astype(np.float32)
    df["rally_id_norm"] = df["rally_id"].astype(np.float32)
    df["next_is_main"]  = ((df["strikeNumber"] + 1) % 2 == 1).astype(np.float32)
    return df


def preprocess(df, encoders=None, player_encoder=None, player_stats=None, sex_stats=None, is_train=True):
    df = df.copy()
    df = df.sort_values(["rally_uid", "strikeNumber"]).reset_index(drop=True)
    df = attach_player_te(df, player_stats, sex_stats)
    df = add_derived(df)
    if is_train:
        encoders = {}
        for c in CAT_COLS_NORMAL:
            uniq = df[c].dropna().unique()
            encoders[c] = {v: i + 2 for i, v in enumerate(uniq)}
        all_players = pd.concat([df["gamePlayerId"], df["gamePlayerOtherId"]]).dropna().unique()
        player_encoder = {v: i + 2 for i, v in enumerate(all_players)}
    for c in CAT_COLS_NORMAL:
        df[c] = df[c].map(encoders[c]).fillna(UNK_IDX).astype(int)
    for c in PLAYER_COLS:
        df[c] = df[c].map(player_encoder).fillna(UNK_IDX).astype(int)
    # V12 fix: normalization scales aligned to real data range
    df["scoreSelf"]     = df["scoreSelf"]     / 15.0
    df["scoreOther"]    = df["scoreOther"]    / 15.0
    df["scoreDiff"]     = df["scoreDiff"]     / 15.0
    df["scoreSum"]      = df["scoreSum"]      / 22.0
    df["strikeNumber"]  = df["strikeNumber"]  / 25.0
    df["rally_id_norm"] = df["rally_id_norm"] / 30.0
    return df, encoders, player_encoder


# ============================================================
# Dataset & Loader (same as V15)
# ============================================================

## CELL 7: Dataset + collate_fn

In [ ]:
class TTDataset(Dataset):
    def __init__(self, df, mode="train"):
        self.mode = mode
        self.samples = []
        K = CFG.rally_end_horizon
        for uid, g in df.groupby("rally_uid", sort=False):
            g = g.sort_values("strikeNumber")
            x_cat_normal = g[CAT_COLS_NORMAL].values.astype(np.int64)
            x_player     = g[PLAYER_COLS].values.astype(np.int64)
            x_num        = g[NUM_COLS].values.astype(np.float32)
            seq_len = len(g)
            if mode == "test":
                self.samples.append({
                    "uid": uid, "x_cat": x_cat_normal, "x_player": x_player, "x_num": x_num,
                    "y_action": -100, "y_point": -100, "y_server": 0.0,
                    "y_rally_continue": np.zeros(K, dtype=np.float32), "length": seq_len,
                    "server_w": 0.0,
                })
            else:
                if seq_len < 2: continue
                y_server = float(g["serverGetPoint"].iloc[0])
                srv_w = 0.0 if (getattr(CFG, "fix_server_leak", False) and "is_old" in g.columns
                                 and int(g["is_old"].iloc[0]) == 1) else 1.0
                action_arr = g["actionId"].values
                point_arr  = g["pointId"].values
                for k in range(1, seq_len):
                    rally_cont = np.zeros(K, dtype=np.float32)
                    for m in range(1, K + 1):
                        rally_cont[m - 1] = 1.0 if seq_len > k + m else 0.0
                    self.samples.append({
                        "uid": uid, "x_cat": x_cat_normal[:k], "x_player": x_player[:k], "x_num": x_num[:k],
                        "y_action": int(action_arr[k]), "y_point": int(point_arr[k]),
                        "y_server": y_server, "y_rally_continue": rally_cont, "length": k,
                        "server_w": srv_w,
                    })
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "uid": s["uid"], "x_cat": torch.from_numpy(np.asarray(s["x_cat"])),
            "x_player": torch.from_numpy(np.asarray(s["x_player"])),
            "x_num": torch.from_numpy(np.asarray(s["x_num"])),
            "y_action": torch.tensor(s["y_action"], dtype=torch.long),
            "y_point": torch.tensor(s["y_point"], dtype=torch.long),
            "y_server": torch.tensor(s["y_server"], dtype=torch.float),
            "y_rally_continue": torch.from_numpy(s["y_rally_continue"]),
            "server_w": torch.tensor(s["server_w"], dtype=torch.float),
            "length": s["length"],
        }


def collate_fn(batch):
    lengths = torch.tensor([max(b["length"], 1) for b in batch], dtype=torch.long)
    x_cat    = nn.utils.rnn.pad_sequence([b["x_cat"]    for b in batch], batch_first=True, padding_value=PAD_IDX)
    x_player = nn.utils.rnn.pad_sequence([b["x_player"] for b in batch], batch_first=True, padding_value=PAD_IDX)
    x_num    = nn.utils.rnn.pad_sequence([b["x_num"]    for b in batch], batch_first=True, padding_value=0.0)
    return (x_cat, x_player, x_num,
            torch.stack([b["y_action"] for b in batch]), torch.stack([b["y_point"]  for b in batch]),
            torch.stack([b["y_server"] for b in batch]), torch.stack([b["y_rally_continue"] for b in batch]),
            lengths, [b["uid"] for b in batch], torch.stack([b["server_w"] for b in batch]))


# ============================================================
# Bi-GRU with pack_padded_sequence (V12 fix kept)
# ============================================================

## CELL 8: NN 模型: Multi-Task Bi-GRU

In [ ]:
class MultiTaskGRUV3(nn.Module):
    def __init__(self, cat_dims, player_dim):
        super().__init__()
        emb_dim = CFG.d_model // 4
        K = CFG.rally_end_horizon
        self.embs = nn.ModuleList([nn.Embedding(v, emb_dim, padding_idx=PAD_IDX) for v in cat_dims])
        self.player_emb = nn.Embedding(player_dim, emb_dim, padding_idx=PAD_IDX)
        n_cat = len(cat_dims) + 2
        total_emb = emb_dim * n_cat
        self.input_proj = nn.Linear(total_emb + len(NUM_COLS), CFG.d_model)
        self.input_norm = nn.LayerNorm(CFG.d_model)
        self.gru = nn.GRU(input_size=CFG.d_model, hidden_size=CFG.d_model,
            num_layers=2, batch_first=True, bidirectional=True, dropout=CFG.dropout)
        hidden_out = CFG.d_model * 2
        self.action_head = nn.Linear(hidden_out, N_ACTION)
        self.point_head  = nn.Linear(hidden_out, N_POINT)
        self.rally_cont_head = nn.Sequential(
            nn.Linear(hidden_out, CFG.d_model), nn.GELU(), nn.Dropout(CFG.dropout),
            nn.Linear(CFG.d_model, K))
        self.server_head = nn.Sequential(
            nn.Linear(hidden_out, CFG.d_model), nn.GELU(), nn.Dropout(CFG.dropout),
            nn.Linear(CFG.d_model, 1))

    def _build_input(self, x_cat, x_player, x_num):
        cat_embs = [self.embs[i](x_cat[:, :, i]) for i in range(x_cat.size(-1))]
        for i in range(x_player.size(-1)):
            cat_embs.append(self.player_emb(x_player[:, :, i]))
        embs = torch.cat(cat_embs, dim=-1)
        x = torch.cat([embs, x_num], dim=-1)
        return self.input_norm(self.input_proj(x))

    def forward(self, x_cat, x_player, x_num, lengths):
        B, T = x_cat.shape[:2]; dev = x_cat.device
        x = self._build_input(x_cat, x_player, x_num)
        lengths_cpu = lengths.detach().cpu()
        packed = pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.gru(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)
        last_idx  = (lengths - 1).clamp(min=0).to(dev)
        batch_idx = torch.arange(B, device=dev)
        last_h    = out[batch_idx, last_idx]
        pad_mask   = (torch.arange(T, device=dev)[None, :] >= lengths[:, None].to(dev))
        mask_float = (~pad_mask).unsqueeze(-1).float()
        pooled     = (out * mask_float).sum(1) / mask_float.sum(1).clamp(min=1e-6)
        return {
            "action":     self.action_head(last_h),
            "point":      self.point_head(last_h),
            "server":     self.server_head(pooled).squeeze(-1),
            "rally_cont": self.rally_cont_head(last_h),
        }


# ============================================================
# Train helpers (same as V15)
# ============================================================

In [ ]:
# ============================================================
# Transformer 多任務模型 (與 MultiTaskGRUV3 同輸入/輸出, 供 ensemble)
# ============================================================
class MultiTaskTransformerV3(nn.Module):
    def __init__(self, cat_dims, player_dim):
        super().__init__()
        emb_dim = CFG.d_model // 4
        K = CFG.rally_end_horizon
        self.embs = nn.ModuleList([nn.Embedding(v, emb_dim, padding_idx=PAD_IDX) for v in cat_dims])
        self.player_emb = nn.Embedding(player_dim, emb_dim, padding_idx=PAD_IDX)
        n_cat = len(cat_dims) + 2
        self.input_proj = nn.Linear(emb_dim * n_cat + len(NUM_COLS), CFG.d_model)
        self.input_norm = nn.LayerNorm(CFG.d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, CFG.max_seq_len + 1, CFG.d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, CFG.d_model))
        nn.init.trunc_normal_(self.cls, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=CFG.d_model, nhead=4, dim_feedforward=CFG.d_model * 2,
            dropout=CFG.dropout, batch_first=True, activation="gelu", norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        h = CFG.d_model
        self.action_head = nn.Linear(h, N_ACTION)
        self.point_head  = nn.Linear(h, N_POINT)
        self.rally_cont_head = nn.Sequential(nn.Linear(h, CFG.d_model), nn.GELU(),
                                             nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, K))
        self.server_head = nn.Sequential(nn.Linear(h, CFG.d_model), nn.GELU(),
                                         nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, 1))

    def _build_input(self, x_cat, x_player, x_num):
        cat_embs = [self.embs[i](x_cat[:, :, i]) for i in range(x_cat.size(-1))]
        for i in range(x_player.size(-1)):
            cat_embs.append(self.player_emb(x_player[:, :, i]))
        x = torch.cat([torch.cat(cat_embs, dim=-1), x_num], dim=-1)
        return self.input_norm(self.input_proj(x))

    def forward(self, x_cat, x_player, x_num, lengths):
        B, T = x_cat.shape[:2]; dev = x_cat.device
        x = self._build_input(x_cat, x_player, x_num)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_emb[:, :T + 1, :]
        pad = (torch.arange(T, device=dev)[None, :] >= lengths[:, None].to(dev))
        kpm = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=dev), pad], dim=1)
        hseq = self.encoder(x, src_key_padding_mask=kpm)
        cls_h = hseq[:, 0]
        last_idx = lengths.clamp(min=1).to(dev)          # +1(cls) -1 = lengths
        last_h = hseq[torch.arange(B, device=dev), last_idx]
        return {"action": self.action_head(last_h), "point": self.point_head(last_h),
                "server": self.server_head(cls_h).squeeze(-1),
                "rally_cont": self.rally_cont_head(cls_h)}


In [ ]:
# ============================================================
# ShuttleNet 啟發版: rally-progress 編碼 + 選手風格 FiLM 融合
# (適配本任務多任務輸出; 輸入/輸出與其他 NN 一致, 供 ensemble)
# ============================================================
class MultiTaskShuttleNetV3(nn.Module):
    def __init__(self, cat_dims, player_dim):
        super().__init__()
        emb_dim = CFG.d_model // 4
        K = CFG.rally_end_horizon
        self.embs = nn.ModuleList([nn.Embedding(v, emb_dim, padding_idx=PAD_IDX) for v in cat_dims])
        self.player_emb = nn.Embedding(player_dim, emb_dim, padding_idx=PAD_IDX)
        n_cat = len(cat_dims) + 2
        self.input_proj = nn.Linear(emb_dim * n_cat + len(NUM_COLS), CFG.d_model)
        self.input_norm = nn.LayerNorm(CFG.d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, CFG.max_seq_len + 1, CFG.d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, CFG.d_model))
        nn.init.trunc_normal_(self.cls, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=CFG.d_model, nhead=4, dim_feedforward=CFG.d_model * 2,
            dropout=CFG.dropout, batch_first=True, activation="gelu", norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        # 選手風格 FiLM: 由兩位選手 embedding 產生 gamma/beta 調變表示
        self.film = nn.Sequential(nn.Linear(emb_dim * 2, CFG.d_model), nn.GELU(),
                                  nn.Linear(CFG.d_model, CFG.d_model * 2))
        h = CFG.d_model
        self.action_head = nn.Linear(h, N_ACTION)
        self.point_head  = nn.Linear(h, N_POINT)
        self.rally_cont_head = nn.Sequential(nn.Linear(h, CFG.d_model), nn.GELU(),
                                             nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, K))
        self.server_head = nn.Sequential(nn.Linear(h, CFG.d_model), nn.GELU(),
                                         nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, 1))

    def _build_input(self, x_cat, x_player, x_num):
        cat_embs = [self.embs[i](x_cat[:, :, i]) for i in range(x_cat.size(-1))]
        for i in range(x_player.size(-1)):
            cat_embs.append(self.player_emb(x_player[:, :, i]))
        x = torch.cat([torch.cat(cat_embs, dim=-1), x_num], dim=-1)
        return self.input_norm(self.input_proj(x))

    def _player_style(self, x_player):
        p = self.player_emb(x_player[:, 0, :])      # 首拍兩位選手 = 該 rally 風格
        p = p.reshape(p.size(0), -1)
        gamma, beta = self.film(p).chunk(2, dim=-1)
        return gamma, beta

    def forward(self, x_cat, x_player, x_num, lengths):
        B, T = x_cat.shape[:2]; dev = x_cat.device
        x = self._build_input(x_cat, x_player, x_num)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_emb[:, :T + 1, :]
        pad = (torch.arange(T, device=dev)[None, :] >= lengths[:, None].to(dev))
        kpm = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=dev), pad], dim=1)
        hseq = self.encoder(x, src_key_padding_mask=kpm)
        cls_h = hseq[:, 0]
        last_idx = lengths.clamp(min=1).to(dev)
        last_h = hseq[torch.arange(B, device=dev), last_idx]
        gamma, beta = self._player_style(x_player)
        last_h = gamma * last_h + beta
        cls_h  = gamma * cls_h + beta
        return {"action": self.action_head(last_h), "point": self.point_head(last_h),
                "server": self.server_head(cls_h).squeeze(-1),
                "rally_cont": self.rally_cont_head(cls_h)}


# ============================================================
# 額外 NN 成員 (opt-in): Bi-LSTM 與 causal TCN, 供 ensemble 多樣性
# 與 MultiTaskGRUV3 同輸入/輸出; 由 OOF 權重搜尋自動決定混入權重(沒幫助則 0)
# ============================================================
class MultiTaskLSTMV3(nn.Module):
    def __init__(self, cat_dims, player_dim):
        super().__init__()
        emb_dim = CFG.d_model // 4; K = CFG.rally_end_horizon
        self.embs = nn.ModuleList([nn.Embedding(v, emb_dim, padding_idx=PAD_IDX) for v in cat_dims])
        self.player_emb = nn.Embedding(player_dim, emb_dim, padding_idx=PAD_IDX)
        n_cat = len(cat_dims) + 2
        self.input_proj = nn.Linear(emb_dim * n_cat + len(NUM_COLS), CFG.d_model)
        self.input_norm = nn.LayerNorm(CFG.d_model)
        self.lstm = nn.LSTM(input_size=CFG.d_model, hidden_size=CFG.d_model,
            num_layers=2, batch_first=True, bidirectional=True, dropout=CFG.dropout)
        h = CFG.d_model * 2
        self.action_head = nn.Linear(h, N_ACTION); self.point_head = nn.Linear(h, N_POINT)
        self.rally_cont_head = nn.Sequential(nn.Linear(h, CFG.d_model), nn.GELU(), nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, K))
        self.server_head = nn.Sequential(nn.Linear(h, CFG.d_model), nn.GELU(), nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, 1))
    def _build_input(self, x_cat, x_player, x_num):
        cat_embs = [self.embs[i](x_cat[:, :, i]) for i in range(x_cat.size(-1))]
        for i in range(x_player.size(-1)): cat_embs.append(self.player_emb(x_player[:, :, i]))
        x = torch.cat([torch.cat(cat_embs, dim=-1), x_num], dim=-1)
        return self.input_norm(self.input_proj(x))
    def forward(self, x_cat, x_player, x_num, lengths):
        B, T = x_cat.shape[:2]; dev = x_cat.device
        x = self._build_input(x_cat, x_player, x_num)
        packed = pack_padded_sequence(x, lengths.detach().cpu(), batch_first=True, enforce_sorted=False)
        out, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True, total_length=T)
        last_idx = (lengths - 1).clamp(min=0).to(dev); bidx = torch.arange(B, device=dev)
        last_h = out[bidx, last_idx]
        pad = (torch.arange(T, device=dev)[None, :] >= lengths[:, None].to(dev))
        mf = (~pad).unsqueeze(-1).float(); pooled = (out * mf).sum(1) / mf.sum(1).clamp(min=1e-6)
        return {"action": self.action_head(last_h), "point": self.point_head(last_h),
                "server": self.server_head(pooled).squeeze(-1), "rally_cont": self.rally_cont_head(last_h)}


class _Chomp1d(nn.Module):
    def __init__(self, chomp): super().__init__(); self.chomp = chomp
    def forward(self, x): return x[:, :, :-self.chomp].contiguous() if self.chomp > 0 else x

class MultiTaskTCNV3(nn.Module):
    def __init__(self, cat_dims, player_dim):
        super().__init__()
        emb_dim = CFG.d_model // 4; K = CFG.rally_end_horizon
        self.embs = nn.ModuleList([nn.Embedding(v, emb_dim, padding_idx=PAD_IDX) for v in cat_dims])
        self.player_emb = nn.Embedding(player_dim, emb_dim, padding_idx=PAD_IDX)
        n_cat = len(cat_dims) + 2
        self.input_proj = nn.Linear(emb_dim * n_cat + len(NUM_COLS), CFG.d_model)
        self.input_norm = nn.LayerNorm(CFG.d_model)
        ch = CFG.d_model; layers = []
        for d in [1, 2, 4]:
            pad = (3 - 1) * d
            layers += [nn.Conv1d(ch, ch, 3, padding=pad, dilation=d), _Chomp1d(pad), nn.GELU(), nn.Dropout(CFG.dropout)]
        self.tcn = nn.Sequential(*layers)
        self.action_head = nn.Linear(ch, N_ACTION); self.point_head = nn.Linear(ch, N_POINT)
        self.rally_cont_head = nn.Sequential(nn.Linear(ch, CFG.d_model), nn.GELU(), nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, K))
        self.server_head = nn.Sequential(nn.Linear(ch, CFG.d_model), nn.GELU(), nn.Dropout(CFG.dropout), nn.Linear(CFG.d_model, 1))
    def _build_input(self, x_cat, x_player, x_num):
        cat_embs = [self.embs[i](x_cat[:, :, i]) for i in range(x_cat.size(-1))]
        for i in range(x_player.size(-1)): cat_embs.append(self.player_emb(x_player[:, :, i]))
        x = torch.cat([torch.cat(cat_embs, dim=-1), x_num], dim=-1)
        return self.input_norm(self.input_proj(x))
    def forward(self, x_cat, x_player, x_num, lengths):
        B, T = x_cat.shape[:2]; dev = x_cat.device
        x = self._build_input(x_cat, x_player, x_num)
        h = self.tcn(x.transpose(1, 2)).transpose(1, 2)
        last_idx = (lengths - 1).clamp(min=0).to(dev); last_h = h[torch.arange(B, device=dev), last_idx]
        pad = (torch.arange(T, device=dev)[None, :] >= lengths[:, None].to(dev))
        mf = (~pad).unsqueeze(-1).float(); pooled = (h * mf).sum(1) / mf.sum(1).clamp(min=1e-6)
        return {"action": self.action_head(last_h), "point": self.point_head(last_h),
                "server": self.server_head(pooled).squeeze(-1), "rally_cont": self.rally_cont_head(last_h)}


## CELL 9: NN 訓練 helper: class weight / server-from-rally / sampler

In [ ]:
def compute_class_weight(labels, n_class):
    counts = np.bincount(labels, minlength=n_class).astype(np.float64)
    counts = np.where(counts == 0, 1.0, counts)
    w = counts.sum() / (n_class * counts)
    return torch.tensor(np.clip(w, 0.3, 5.0), dtype=torch.float)


def server_from_rally_continue(prefix_len, p_continue, server_raw, K=CFG.rally_end_horizon):
    pc = np.copy(p_continue)
    for i in range(1, K):
        if pc[i] > pc[i - 1]: pc[i] = pc[i - 1]
    pc = np.clip(pc, 0.0, 1.0)
    p_end = np.zeros(K, dtype=np.float64)
    p_end[0] = 1.0 - pc[0]
    for m in range(2, K + 1):
        p_end[m - 1] = max(pc[m - 2] - pc[m - 1], 0.0)
    p_beyond = pc[K - 1]
    p_server = 0.0
    for m in range(1, K + 1):
        total_len = prefix_len + m
        wins = 1.0 if (total_len % 2 == 0) else 0.0
        p_server += p_end[m - 1] * wins
    p_server += p_beyond * server_raw
    return p_server


def make_prefix_sampler(dataset, test_dist, total_te):
    train_dist = Counter([s["length"] for s in dataset.samples])
    total_tr = sum(train_dist.values())
    weights = []
    for s in dataset.samples:
        k = s["length"]
        p_te = test_dist.get(k, 0) / total_te
        p_tr = train_dist[k] / total_tr if k in train_dist else 1.0 / total_tr
        w = (p_te / p_tr) if p_tr > 0 else 0.3
        weights.append(min(max(w, 0.3), 3.0))
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.double),
                                  num_samples=len(weights), replacement=True)

## CELL 10: NN 訓練主迴圈: train / evaluate / predict

In [ ]:
def train_one_dl_fold(tr_df, va_df, fold_idx, seed, encoders, player_encoder, test_dist, total_te,
                        model_class=MultiTaskGRUV3):
    seed_everything(seed)
    train_ds = TTDataset(tr_df, mode="train")
    val_ds   = TTDataset(va_df, mode="train")
    print(f"  fold {fold_idx} seed {seed} | tr {len(train_ds)} va {len(val_ds)}")
    sampler = make_prefix_sampler(train_ds, test_dist, total_te)
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, sampler=sampler,
                                collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                                collate_fn=collate_fn, num_workers=2, pin_memory=True)

    cat_dims = [len(encoders[c]) + 2 for c in CAT_COLS_NORMAL]
    base = model_class(cat_dims, len(player_encoder) + 2)
    model = nn.DataParallel(base).to(device) if torch.cuda.device_count() > 1 else base.to(device)

    w_a = compute_class_weight(np.array([s["y_action"] for s in train_ds.samples]), N_ACTION).to(device) if CFG.use_class_weight else None
    w_p = compute_class_weight(np.array([s["y_point"]  for s in train_ds.samples]), N_POINT).to(device) if CFG.use_class_weight else None
    ce_a = nn.CrossEntropyLoss(weight=w_a, ignore_index=-100, label_smoothing=CFG.label_smoothing)
    ce_p = nn.CrossEntropyLoss(weight=w_p, ignore_index=-100, label_smoothing=CFG.label_smoothing)
    bce, bce_rc = nn.BCEWithLogitsLoss(), nn.BCEWithLogitsLoss()
    srv_bce = nn.BCEWithLogitsLoss(reduction="none")
    rc_bce  = nn.BCEWithLogitsLoss(reduction="none")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=CFG.min_lr)
    scaler = torch.amp.GradScaler("cuda") if hasattr(torch.amp, "GradScaler") and USE_AMP else (torch.cuda.amp.GradScaler() if USE_AMP else None)
    autocast_ctx = lambda: torch.amp.autocast("cuda") if hasattr(torch.amp, "autocast") else torch.cuda.amp.autocast()

    best_score, best_state, no_improve = -1.0, None, 0
    for epoch in range(CFG.epochs):
        model.train(); total_loss, n_batch = 0.0, 0
        for x_cat, x_player, x_num, y_a, y_p, y_s, y_rc, lengths, _, sw in train_loader:
            x_cat = x_cat.to(device, non_blocking=True); x_player = x_player.to(device, non_blocking=True)
            x_num = x_num.to(device, non_blocking=True)
            y_a, y_p, y_s, y_rc, lengths = y_a.to(device), y_p.to(device), y_s.to(device), y_rc.to(device), lengths.to(device)
            sw = sw.to(device)

            if CFG.player_dropout_p > 0:
                drop_mask = (torch.rand(x_player.shape, device=device) < CFG.player_dropout_p)
                x_player = torch.where(drop_mask & (x_player != PAD_IDX),
                                         torch.full_like(x_player, UNK_IDX), x_player)
            optimizer.zero_grad()
            if USE_AMP:
                with autocast_ctx():
                    out = model(x_cat, x_player, x_num, lengths)
                    loss = (CFG.w_action * ce_a(out["action"], y_a) + CFG.w_point * ce_p(out["point"], y_p)
                          + CFG.w_server * (srv_bce(out["server"], y_s) * sw).sum() / sw.sum().clamp(min=1.0) + CFG.w_rally_end * (rc_bce(out["rally_cont"], y_rc).mean(dim=1) * sw).sum() / sw.sum().clamp(min=1.0))
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                scaler.step(optimizer); scaler.update()
            else:
                out = model(x_cat, x_player, x_num, lengths)
                loss = (CFG.w_action * ce_a(out["action"], y_a) + CFG.w_point * ce_p(out["point"], y_p)
                      + CFG.w_server * (srv_bce(out["server"], y_s) * sw).sum() / sw.sum().clamp(min=1.0) + CFG.w_rally_end * (rc_bce(out["rally_cont"], y_rc).mean(dim=1) * sw).sum() / sw.sum().clamp(min=1.0))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                optimizer.step()
            total_loss += loss.item(); n_batch += 1
        scheduler.step()

        score, f1a, f1p, auc = evaluate(model, val_loader)
        print(f"  Ep {epoch+1:02d} | loss {total_loss/max(n_batch,1):.4f} | "
              f"score {score:.4f} | f1a {f1a:.4f} f1p {f1p:.4f} auc {auc:.4f}")
        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= CFG.patience:
                print(f"  Early stop at epoch {epoch+1} (best {best_score:.4f})")
                break
    model.load_state_dict(best_state)
    return model, best_score


def evaluate(model, loader):
    model.eval()
    a_t, a_p, p_t, p_p, s_t, s_p, lens_all, rc_all = [], [], [], [], [], [], [], []
    with torch.no_grad():
        for x_cat, x_player, x_num, y_a, y_p, y_s, y_rc, lengths, _, _ in loader:
            out = model(x_cat.to(device), x_player.to(device), x_num.to(device), lengths.to(device))
            pa = F.softmax(out["action"], dim=-1).cpu().numpy()
            pp = F.softmax(out["point"],  dim=-1).cpu().numpy()
            ps = torch.sigmoid(out["server"]).cpu().numpy()
            prc = torch.sigmoid(out["rally_cont"]).cpu().numpy()
            a_t.extend(y_a.numpy()); a_p.extend(pa.argmax(-1))
            p_t.extend(y_p.numpy()); p_p.extend(pp.argmax(-1))
            s_t.extend(y_s.numpy()); s_p.extend(ps)
            lens_all.extend(lengths.numpy()); rc_all.extend(prc)
    f1a = f1_score(a_t, a_p, average="macro", zero_division=0)
    f1p = f1_score(p_t, p_p, average="macro", zero_division=0)
    s_blend = np.array([server_from_rally_continue(L, rc, sr) for L, rc, sr in zip(lens_all, rc_all, s_p)])
    try:    auc = roc_auc_score(s_t, s_blend)
    except: auc = 0.5
    return 0.4 * f1a + 0.4 * f1p + 0.2 * auc, f1a, f1p, auc


def predict_with_keys(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for x_cat, x_player, x_num, y_a, y_p, y_s, y_rc, lengths, uids, _ in loader:
            out = model(x_cat.to(device), x_player.to(device), x_num.to(device), lengths.to(device))
            pa = F.softmax(out["action"], dim=-1).cpu().numpy()
            pp = F.softmax(out["point"],  dim=-1).cpu().numpy()
            ps = torch.sigmoid(out["server"]).cpu().numpy()
            prc = torch.sigmoid(out["rally_cont"]).cpu().numpy()
            for u, k, a, p, s, rc, ya, yp, yss in zip(uids, lengths.numpy(), pa, pp, ps, prc,
                                                         y_a.numpy(), y_p.numpy(), y_s.numpy()):
                rows.append((u, int(k), a, p, server_from_rally_continue(int(k), rc, s),
                              int(ya), int(yp), float(yss)))
    return rows


# ============================================================
# VV3 tabular features = V15 base + 20 select VV2A new features
# ============================================================

## CELL 11: GBDT 特徵工程: build_tabular_features (VV3 的 68 features)

In [ ]:
def build_tabular_features(df_full, mode, cond_stats, act_stats, global_stats,
                            bigram_stats, bigram_global):
    rows = []
    for uid, g in df_full.groupby("rally_uid", sort=False):
        g = g.sort_values("strikeNumber"); L = len(g)
        if mode == "train" and L < 2: continue
        ks = list(range(1, L)) if mode == "train" else [L]
        a_arr  = g["actionId"].values; p_arr = g["pointId"].values
        h_arr  = g["handId"].values; s1_arr = g["strengthId"].values
        sp_arr = g["spinId"].values; pos_arr = g["positionId"].values
        sk_arr = g["strikeId"].values
        score_self = g["scoreSelf"].values; score_other = g["scoreOther"].values
        first = g.iloc[0]

        for k in ks:
            cond = lookup_conditional_pt(pos_arr[k - 1], a_arr[k - 1],
                                          cond_stats, act_stats, global_stats)
            pre_a = a_arr[:k]; pre_h = h_arr[:k]

            last_a = int(a_arr[k - 1]); last_h = int(h_arr[k - 1])
            last_sp = int(sp_arr[k - 1]); last_pos = int(pos_arr[k - 1])
            last_s1 = int(s1_arr[k - 1])
            p2_a = int(a_arr[k - 2]) if k >= 2 else -1
            p3_a = int(a_arr[k - 3]) if k >= 3 else -1
            p4_a = int(a_arr[k - 4]) if k >= 4 else -1

            # === VV3 NEW: bigram aggregate (not raw code; raw code blocked) ===
            bg = lookup_bigram_a(p2_a, last_a, bigram_stats, bigram_global) if k >= 2 else bigram_global

            # === VV3 NEW: low-cardinality cross only ===
            cross_a_h   = last_a * 3 + last_h      # 57 codes
            cross_h_pos = last_h * 4 + last_pos    # 12 codes

            # === VV3 NEW: score situation ===
            cur_self = int(score_self[k - 1]); cur_other = int(score_other[k - 1])
            is_deuce     = int(cur_self >= 10 and cur_other >= 10 and abs(cur_self - cur_other) <= 1)
            is_set_point = int(max(cur_self, cur_other) >= 10 and abs(cur_self - cur_other) >= 1)
            is_lead      = int(cur_self > cur_other)
            max_score    = max(cur_self, cur_other)
            score_close  = int(abs(cur_self - cur_other) <= 2)

            # === VV3 NEW: recent pattern ===
            recent_window  = min(3, k)
            recent_attack  = int(sum(1 for a in pre_a[-3:] if a in ATTACK_ACTIONS))
            recent_control = int(sum(1 for a in pre_a[-3:] if a in CONTROL_ACTIONS))
            consec_same_h = 1
            for i in range(len(pre_h) - 2, -1, -1):
                if pre_h[i] == pre_h[-1]: consec_same_h += 1
                else: break
            alt_last2 = int(k >= 2 and pre_h[-1] != pre_h[-2])

            # === VV3 NEW: distance to last attack ===
            last_attack_dist = -1
            for i in range(len(pre_a) - 1, -1, -1):
                if pre_a[i] in ATTACK_ACTIONS:
                    last_attack_dist = (k - 1) - i
                    break

            row = {
                "rally_uid": uid, "k": k, "match": first["match"], "sex": int(first["sex"]),
                "numberGame": int(first["numberGame"]), "rally_id": int(first["rally_id"]),
                "scoreSelf": cur_self, "scoreOther": cur_other,
                "scoreDiff": cur_self - cur_other, "scoreSum": cur_self + cur_other,
                "playerSelf": first["gamePlayerId"], "playerOther": first["gamePlayerOtherId"],
                "next_is_main": int((k + 1) % 2 == 1),
                # base stroke features (from V15)
                "last_action": last_a, "last_point": int(p_arr[k - 1]), "last_hand": last_h,
                "last_strength": last_s1, "last_spin": last_sp,
                "last_position": last_pos, "last_strikeId": int(sk_arr[k - 1]),
                "p2_action": p2_a, "p2_point": int(p_arr[k - 2]) if k >= 2 else -1,
                "p2_hand": int(h_arr[k - 2]) if k >= 2 else -1,
                "p2_spin": int(sp_arr[k - 2]) if k >= 2 else -1,
                "serve_action": int(a_arr[0]), "serve_point": int(p_arr[0]),
                "serve_hand": int(h_arr[0]), "serve_spin": int(sp_arr[0]),
                "serve_strength": int(s1_arr[0]),
                "cond_pt_top1_p": cond["top1_p"], "cond_pt_top1_id": cond["top1_id"], "cond_pt_ent": cond["ent"],
                # VV3 NEW (20 features, all VV2A-safe ones)
                "p3_action": p3_a, "p3_point": int(p_arr[k - 3]) if k >= 3 else -1,
                "p3_hand": int(h_arr[k - 3]) if k >= 3 else -1,
                "p4_action": p4_a,
                "bg_top1_p": bg["top1_p"], "bg_top1_id": bg["top1_id"], "bg_ent": bg["ent"],
                "cross_a_h": cross_a_h, "cross_h_pos": cross_h_pos,
                "is_deuce": is_deuce, "is_set_point": is_set_point, "is_lead": is_lead,
                "max_score": max_score, "score_close": score_close,
                "recent_window": recent_window,
                "recent_attack": recent_attack, "recent_control": recent_control,
                "consec_same_h": consec_same_h, "alt_last2": alt_last2,
                "last_attack_dist": last_attack_dist,
            }
            for c in PLAYER_TE_COLS: row[c] = first[c]
            if mode == "train":
                row["target_action"] = int(a_arr[k])
                row["target_point"]  = int(p_arr[k])
                row["target_server"] = float(first["serverGetPoint"])
            rows.append(row)
    return pd.DataFrame(rows)


# ============================================================
# GBDT fits (with seed parameter for multi-seed bagging)
# ============================================================

## CELL 12: GBDT 單模型 fit (LGB / XGB / Cat)

In [ ]:
def fit_gbdt_classifier(model_kind, n_estimators, n_class, seed, X_tr, y_tr, w_tr, X_va, X_te, balanced=True):
    if model_kind == "lgb":
        params = dict(n_estimators=n_estimators, learning_rate=0.05, num_leaves=63, min_child_samples=30,
                       verbose=-1, random_state=seed, n_jobs=-1, colsample_bytree=0.8, subsample=0.8, subsample_freq=1)
        if balanced: params["class_weight"] = "balanced"
        clf = lgb.LGBMClassifier(**params); clf.fit(X_tr, y_tr, sample_weight=w_tr)
    elif model_kind == "xgb":
        X_tr_use, y_tr_use = X_tr, np.asarray(y_tr).copy()
        sample_w = w_tr.copy() if w_tr is not None else np.ones(len(y_tr))
        if n_class > 2:
            unique_y = set(int(v) for v in np.unique(y_tr_use))
            miss = sorted(c for c in range(n_class) if c not in unique_y)
            if miss:
                pad_X = X_tr_use.iloc[[0] * len(miss)].reset_index(drop=True)
                pad_y = np.array(miss, dtype=y_tr_use.dtype)
                pad_w = np.ones(len(miss)) * 1e-6
                X_tr_use = pd.concat([X_tr_use, pad_X], ignore_index=True)
                y_tr_use = np.concatenate([y_tr_use, pad_y])
                sample_w = np.concatenate([sample_w, pad_w])
            if balanced:
                u2, cnts = np.unique(y_tr_use, return_counts=True)
                class_w = np.ones(n_class)
                for cls, cw in zip(u2, cnts.sum() / (len(u2) * cnts)):
                    class_w[int(cls)] = min(max(cw, 0.3), 5.0)
                sample_w = sample_w * np.array([class_w[int(y)] for y in y_tr_use])
        clf = xgb.XGBClassifier(n_estimators=n_estimators, learning_rate=0.05, max_depth=6, min_child_weight=5,
                                 subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, verbosity=0,
                                 tree_method="hist", device="cuda" if torch.cuda.is_available() else "cpu",
                                 eval_metric="mlogloss" if n_class > 2 else "logloss")
        clf.fit(X_tr_use, y_tr_use, sample_weight=sample_w)
    elif model_kind == "cat":
        X_tr_use, y_tr_use = X_tr, np.asarray(y_tr).copy()
        sample_w = w_tr.copy() if w_tr is not None else np.ones(len(y_tr))
        if n_class > 2:
            unique_y = set(int(v) for v in np.unique(y_tr_use))
            miss = sorted(c for c in range(n_class) if c not in unique_y)
            if miss:
                pad_X = X_tr_use.iloc[[0] * len(miss)].reset_index(drop=True)
                pad_y = np.array(miss, dtype=y_tr_use.dtype)
                pad_w = np.ones(len(miss)) * 1e-6
                X_tr_use = pd.concat([X_tr_use, pad_X], ignore_index=True)
                y_tr_use = np.concatenate([y_tr_use, pad_y])
                sample_w = np.concatenate([sample_w, pad_w])
        class_weights = None
        if balanced:
            u2, cnts = np.unique(y_tr_use, return_counts=True)
            cw_dict = {int(cls): min(max(cnts.sum() / (len(u2) * cnt), 0.3), 5.0)
                       for cls, cnt in zip(u2, cnts)}
            if n_class > 2:
                class_weights = [cw_dict.get(c, 1.0) for c in range(n_class)]
            else:
                class_weights = [cw_dict.get(0, 1.0), cw_dict.get(1, 1.0)]
        task_type = "GPU" if torch.cuda.is_available() else "CPU"
        cat_kwargs = dict(iterations=n_estimators, learning_rate=0.05, depth=6, random_seed=seed,
                          verbose=0, allow_writing_files=False, task_type=task_type)
        if class_weights is not None: cat_kwargs["class_weights"] = class_weights
        clf = CatBoostClassifier(**cat_kwargs)
        try: clf.fit(X_tr_use, y_tr_use, sample_weight=sample_w)
        except Exception as e:
            print(f"  CatBoost GPU failed ({e}), retry CPU")
            cat_kwargs["task_type"] = "CPU"
            clf = CatBoostClassifier(**cat_kwargs); clf.fit(X_tr_use, y_tr_use, sample_weight=sample_w)
    elif model_kind in ("rf", "et"):
        X_tr = X_tr.fillna(-1); X_va = X_va.fillna(-1); X_te = X_te.fillna(-1)  # 袋裝樹不吃 NaN, 防呆
        # 袋裝樹 (RandomForest / ExtraTrees): 對未見選手泛化較好, 加入 action GBDT blend 增加多樣性
        Cls = RandomForestClassifier if model_kind == "rf" else ExtraTreesClassifier
        n_trees = 400 if model_kind == "rf" else 500
        clf = Cls(n_estimators=n_trees, min_samples_leaf=(5 if model_kind == "rf" else 4),
                  max_features="sqrt", n_jobs=-1, random_state=seed,
                  class_weight=("balanced" if balanced else None))
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
    else: raise ValueError(model_kind)
    pa_va_full = np.zeros((len(X_va), n_class))
    for i, c in enumerate(clf.classes_): pa_va_full[:, int(c)] = clf.predict_proba(X_va)[:, i]
    pa_te_full = np.zeros((len(X_te), n_class))
    for i, c in enumerate(clf.classes_): pa_te_full[:, int(c)] = clf.predict_proba(X_te)[:, i]
    return pa_va_full, pa_te_full


def fit_gbdt_binary(model_kind, n_estimators, seed, X_tr, y_tr, w_tr, X_va, X_te):
    if model_kind == "lgb":
        clf = lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=0.05, num_leaves=63,
                                  min_child_samples=30, verbose=-1, random_state=seed, n_jobs=-1,
                                  colsample_bytree=0.8, subsample=0.8, subsample_freq=1)
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
    elif model_kind == "xgb":
        clf = xgb.XGBClassifier(n_estimators=n_estimators, learning_rate=0.05, max_depth=6, min_child_weight=5,
                                 subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, verbosity=0,
                                 tree_method="hist", device="cuda" if torch.cuda.is_available() else "cpu",
                                 eval_metric="logloss")
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
    elif model_kind == "cat":
        task_type = "GPU" if torch.cuda.is_available() else "CPU"
        cat_kwargs = dict(iterations=n_estimators, learning_rate=0.05, depth=6, random_seed=seed,
                          verbose=0, allow_writing_files=False, task_type=task_type)
        clf = CatBoostClassifier(**cat_kwargs)
        try: clf.fit(X_tr, y_tr, sample_weight=w_tr)
        except Exception:
            cat_kwargs["task_type"] = "CPU"
            clf = CatBoostClassifier(**cat_kwargs); clf.fit(X_tr, y_tr, sample_weight=w_tr)
    return clf.predict_proba(X_va)[:, 1], clf.predict_proba(X_te)[:, 1]


# ============================================================
# VV3: GBDT track with multi-seed bagging (key change from V15)
# ============================================================

## CELL 13: GBDT track: 5-fold × 3-seed bagging

In [ ]:
def train_gbdt_track(train_raw, test_raw, player_stats, sex_stats, cond_stats, act_stats, global_stats,
                       bigram_stats, bigram_global, test_dist, total_te, label="GBDT", old_uids=None):
    print("\n" + "=" * 60); print(f"{label} (multi-seed bagging, n_seeds={CFG.n_seeds_gbdt})"); print("=" * 60)
    tr = attach_player_te(train_raw.copy(), player_stats, sex_stats)
    te = attach_player_te(test_raw.copy(),  player_stats, sex_stats)
    full_tab = build_tabular_features(tr, mode="train",
                                        cond_stats=cond_stats, act_stats=act_stats, global_stats=global_stats,
                                        bigram_stats=bigram_stats, bigram_global=bigram_global)
    test_tab = build_tabular_features(te, mode="test",
                                        cond_stats=cond_stats, act_stats=act_stats, global_stats=global_stats,
                                        bigram_stats=bigram_stats, bigram_global=bigram_global)
    print(f"  full_tab: {len(full_tab)} rows × {len(full_tab.columns)} cols")
    old_uids = old_uids or set()
    full_tab["is_old"] = full_tab["rally_uid"].isin(old_uids).astype(int)

    train_k_dist = Counter(full_tab["k"].values); total_tr = sum(train_k_dist.values())
    full_tab["w"] = full_tab["k"].map(
        lambda k: min(max((test_dist.get(k, 0) / total_te) /
                          (train_k_dist[k] / total_tr if k in train_k_dist else 1.0 / total_tr), 0.3), 3.0)
        if (train_k_dist[k] / total_tr if k in train_k_dist else 1.0 / total_tr) > 0 else 0.3)

    base_feat = [c for c in full_tab.columns if c not in
                  ["rally_uid", "match", "target_action", "target_point", "target_server", "w", "is_old"]]
    use_chain = getattr(CFG, "use_chain", False)
    print(f"  feature count: {len(base_feat)}" + ("  (+chain: action→point)" if use_chain else ""))

    if CFG.use_cat:
        model_kinds = ["lgb", "xgb", "cat"]
        blend_w = (CFG.lgb_weight_in_gbdt, CFG.xgb_weight_in_gbdt, CFG.cat_weight_in_gbdt)
    elif CFG.use_xgb:
        model_kinds = ["lgb", "xgb"]; blend_w = (1 - CFG.xgb_weight_in_gbdt, CFG.xgb_weight_in_gbdt)
    else:
        model_kinds = ["lgb"]; blend_w = (1.0,)
    seeds_gbdt = [42 + i * 13 for i in range(CFG.n_seeds_gbdt)]
    print(f"  model blend: {list(zip(model_kinds, blend_w))}, GBDT seeds: {seeds_gbdt}")
    print(f"  total fits/task: {CFG.n_folds * len(model_kinds) * CFG.n_seeds_gbdt}")

    oof_a = np.zeros((len(full_tab), N_ACTION)); oof_p = np.zeros((len(full_tab), N_POINT))
    oof_s = np.zeros(len(full_tab)); oof_filled = np.zeros(len(full_tab), dtype=bool)
    sum_te_a = np.zeros((len(test_tab), N_ACTION)); sum_te_p = np.zeros((len(test_tab), N_POINT))
    sum_te_s = np.zeros(len(test_tab)); n_avg = 0
    gbdt_fold_iter = list(GroupKFold(n_splits=CFG.n_folds).split(full_tab, groups=full_tab["match"]))

    def blend_classifier(target_col, n_class, X_tr, w_tr, X_va, X_te, fold, kinds=None, weights=None):
        kinds = model_kinds if kinds is None else kinds
        weights = blend_w if weights is None else weights
        pv = np.zeros((len(X_va), n_class)); pt = np.zeros((len(X_te), n_class))
        for i, kind in enumerate(kinds):
            va_avg = np.zeros((len(X_va), n_class)); te_avg = np.zeros((len(X_te), n_class))
            seeds_use = [seeds_gbdt[0]] if kind in ("rf", "et") else seeds_gbdt
            for s in seeds_use:
                p_va, p_te = fit_gbdt_classifier(kind, CFG.n_estimators, n_class, s + fold * 100,
                                                  X_tr, target_col, w_tr, X_va, X_te, balanced=True)
                va_avg += p_va / len(seeds_use); te_avg += p_te / len(seeds_use)
            pv += weights[i] * va_avg; pt += weights[i] * te_avg
        return pv, pt

    # ===== PASS 1: action + server (不依賴 chain) =====
    for fold, (tr_idx, va_idx) in enumerate(gbdt_fold_iter):
        tr_real = full_tab.iloc[tr_idx]; va_real = full_tab.iloc[va_idx]
        X_tr = tr_real[base_feat]; X_va = va_real[base_feat]; X_te = test_tab[base_feat]
        w_tr = tr_real["w"].values
        if getattr(CFG, "use_rf_et_action", True) and CFG.use_cat:
            act_kinds   = ["lgb", "xgb", "cat", "rf", "et"]
            act_weights = (0.30, 0.20, 0.20, 0.10, 0.20)   # rf+et=0.30 (這版產出乾淨 LB 0.3910)
        else:
            act_kinds, act_weights = model_kinds, blend_w
        pa_va, pa_te = blend_classifier(tr_real["target_action"].values, N_ACTION, X_tr, w_tr, X_va, X_te, fold,
                                        kinds=act_kinds, weights=act_weights)
        ps_va_blend = np.zeros(len(va_real)); ps_te_blend = np.zeros(len(test_tab))
        srv_m = (tr_real["is_old"].values == 0) if getattr(CFG, "fix_server_leak", False) else np.ones(len(tr_real), dtype=bool)
        X_tr_srv = X_tr[srv_m]; y_srv = tr_real["target_server"].values[srv_m]; w_srv = w_tr[srv_m]
        for i, kind in enumerate(model_kinds):
            ps_va_avg = np.zeros(len(va_real)); ps_te_avg = np.zeros(len(test_tab))
            for s in seeds_gbdt:
                ps_va, ps_te = fit_gbdt_binary(kind, CFG.n_estimators, s + fold * 100,
                                                X_tr_srv, y_srv, w_srv, X_va, X_te)
                ps_va_avg += ps_va / len(seeds_gbdt); ps_te_avg += ps_te / len(seeds_gbdt)
            ps_va_blend += blend_w[i] * ps_va_avg; ps_te_blend += blend_w[i] * ps_te_avg
        oof_a[va_idx] = pa_va; oof_s[va_idx] = ps_va_blend; oof_filled[va_idx] = True
        sum_te_a += pa_te; sum_te_s += ps_te_blend; n_avg += 1

    # ===== chain 特徵: 用 action 的 OOF (訓練) / 平均 (測試), 不洩漏 =====
    def chain_cols(prob):
        return prob.argmax(1), prob.max(1), -(prob * np.log(prob + 1e-9)).sum(1)
    point_feat = base_feat
    if use_chain:
        cp, ct, ce = chain_cols(oof_a)
        full_tab = full_tab.assign(chain_act_pred=cp, chain_act_top1=ct, chain_act_ent=ce)
        cp, ct, ce = chain_cols(sum_te_a / n_avg)
        test_tab = test_tab.assign(chain_act_pred=cp, chain_act_top1=ct, chain_act_ent=ce)
        point_feat = base_feat + ["chain_act_pred", "chain_act_top1", "chain_act_ent"]

    # ===== PASS 2: point (吃 chain 特徵), 用相同 folds =====
    for fold, (tr_idx, va_idx) in enumerate(gbdt_fold_iter):
        tr_real = full_tab.iloc[tr_idx]; va_real = full_tab.iloc[va_idx]
        X_tr = tr_real[point_feat]; X_va = va_real[point_feat]; X_te = test_tab[point_feat]
        w_tr = tr_real["w"].values
        pp_va, pp_te = blend_classifier(tr_real["target_point"].values, N_POINT, X_tr, w_tr, X_va, X_te, fold)
        oof_p[va_idx] = pp_va; sum_te_p += pp_te

    # ===== 逐 fold 指標 (從 oof 重算) =====
    f1a_list, f1p_list, auc_list = [], [], []
    for fold, (tr_idx, va_idx) in enumerate(gbdt_fold_iter):
        ya = full_tab["target_action"].values[va_idx]; yp = full_tab["target_point"].values[va_idx]
        ys = full_tab["target_server"].values[va_idx]
        f1a = f1_score(ya, oof_a[va_idx].argmax(1), average="macro", zero_division=0)
        f1p = f1_score(yp, oof_p[va_idx].argmax(1), average="macro", zero_division=0)
        try: auc = roc_auc_score(ys, oof_s[va_idx])
        except: auc = 0.5
        print(f"  {label} fold {fold}: f1a={f1a:.4f} f1p={f1p:.4f} auc={auc:.4f} "
              f"score={0.4*f1a + 0.4*f1p + 0.2*auc:.4f}")
        f1a_list.append(f1a); f1p_list.append(f1p); auc_list.append(auc)
    print(f"\n  {label} mean: f1a={np.mean(f1a_list):.4f} f1p={np.mean(f1p_list):.4f} "
          f"auc={np.mean(auc_list):.4f} "
          f"score={0.4*np.mean(f1a_list) + 0.4*np.mean(f1p_list) + 0.2*np.mean(auc_list):.4f}")
    return {
        "oof_action": oof_a, "oof_point": oof_p, "oof_server": oof_s, "oof_filled": oof_filled,
        "oof_label_action": full_tab["target_action"].values,
        "oof_label_point":  full_tab["target_point"].values,
        "oof_label_server": full_tab["target_server"].values,
        "oof_uid_k": list(zip(full_tab["rally_uid"].values, full_tab["k"].values)),
        "te_uid_k": list(zip(test_tab["rally_uid"].values, test_tab["k"].values)),
        "te_action": sum_te_a / n_avg, "te_point": sum_te_p / n_avg, "te_server": sum_te_s / n_avg,
    }


# ============================================================
# VV3: Weight search with expanded ±0.20 range
# ============================================================

## CELL 14: Ensemble 權重搜尋

In [ ]:
def search_optimal_weights(nn_oof_a, nn_oof_p, nn_oof_s,
                            gbdt_oof_a, gbdt_oof_p, gbdt_oof_s,
                            label_a, label_p, label_s,
                            base_weights=(0.70, 0.70, 0.30),
                            mask=None):
    if mask is None: mask = np.ones(len(label_a), dtype=bool)

    def _score(wa, wp, ws):
        blend_a = (1 - wa) * nn_oof_a + wa * gbdt_oof_a
        blend_p = (1 - wp) * nn_oof_p + wp * gbdt_oof_p
        blend_s = (1 - ws) * nn_oof_s + ws * gbdt_oof_s
        f1a = f1_score(label_a[mask], blend_a[mask].argmax(1), average="macro", zero_division=0)
        f1p = f1_score(label_p[mask], blend_p[mask].argmax(1), average="macro", zero_division=0)
        try: auc = roc_auc_score(label_s[mask], blend_s[mask])
        except: auc = 0.5
        return 0.4 * f1a + 0.4 * f1p + 0.2 * auc, f1a, f1p, auc

    base_score, base_f1a, base_f1p, base_auc = _score(*base_weights)
    print(f"\n=== Ensemble Weight Search (radius=±{CFG.weight_search_radius}, step={CFG.weight_search_step}) ===")
    print(f"  Base {base_weights}: score={base_score:.4f} (f1a={base_f1a:.4f}, f1p={base_f1p:.4f}, auc={base_auc:.4f})")

    # VV3: build wider grid ±0.20, step 0.05 → up to 9 levels per axis
    def _grid(center):
        rad = CFG.weight_search_radius; step = CFG.weight_search_step
        levels = []
        v = -rad
        while v <= rad + 1e-9:
            levels.append(round(center + v, 4))
            v += step
        return [max(0.0, min(1.0, x)) for x in levels]
    grid = {"action": _grid(base_weights[0]), "point": _grid(base_weights[1]), "server": _grid(base_weights[2])}
    print(f"  grid sizes: action={len(grid['action'])}, point={len(grid['point'])}, server={len(grid['server'])} "
          f"= {len(grid['action']) * len(grid['point']) * len(grid['server'])} combos")

    best_score, best_w, best_bk = base_score, base_weights, (base_f1a, base_f1p, base_auc)
    for wa in grid["action"]:
        for wp in grid["point"]:
            for ws in grid["server"]:
                s, f1a, f1p, auc = _score(wa, wp, ws)
                if s > best_score:
                    best_score, best_w, best_bk = s, (wa, wp, ws), (f1a, f1p, auc)
    improvement = best_score - base_score
    print(f"  Best {best_w}: score={best_score:.4f} (f1a={best_bk[0]:.4f}, f1p={best_bk[1]:.4f}, auc={best_bk[2]:.4f})")
    print(f"  Improvement: {improvement:+.4f}")
    if improvement > CFG.weight_search_overfit_threshold:
        print(f"  ⚠️  improvement > {CFG.weight_search_overfit_threshold:.3f}, may overfit — fallback to base.")
        return base_weights, base_score, base_score
    elif improvement <= 0:
        return base_weights, base_score, base_score
    else:
        return best_w, best_score, base_score


# ============================================================
# Main
# ============================================================

## CELL 15: Position Masking (+0.013 LB 的關鍵)

In [ ]:
# ============================================================
# CELL 15: Position-Aware Class Masking (the +0.013 LB trick)
# ============================================================
# 核心洞察: macro F1 的分母 = (y_true ∪ y_pred) 的 class 數。
# 發球類別 (15-18) 在 stroke 2+ 永遠不可能是 target,但模型偶爾會誤測。
# 把這些不可能的 class 機率歸零 → 從 macro F1 分母移除 → 分數提升。
# 已驗證: VV3 0.3800 → VV3+mask 0.3929 (+0.0129 LB)

def compute_position_mask(combined_raw, n_action=N_ACTION, min_count=5):
    """從訓練資料統計: 每個 prefix 長度 k, 哪些 action class 從未/極少出現為 target。"""
    sorted_df = combined_raw.sort_values(["rally_uid", "strikeNumber"])
    stroke_counts = {}
    for uid, g in sorted_df.groupby("rally_uid"):
        if len(g) < 2:
            continue
        a = g["actionId"].values
        for pos in range(1, len(a)):
            ts = pos + 1            # target stroke number
            cls = int(a[pos])
            if ts not in stroke_counts:
                stroke_counts[ts] = Counter()
            stroke_counts[ts][cls] += 1
    mask_dict = {}
    for ts in range(2, 30):
        if ts not in stroke_counts:
            continue
        k = ts - 1                  # prefix length
        c = stroke_counts[ts]
        blocked = set(cls for cls in range(n_action) if c.get(cls, 0) < min_count)
        if blocked:
            mask_dict[k] = blocked
    return mask_dict


def apply_action_mask(ens_te_a, test_uids_order, test_raw, mask_dict, n_action=N_ACTION):
    """對 test 的 action 機率套用 position mask, 把不可能的 class 歸零後重新正規化。"""
    test_uid_lens = test_raw.groupby("rally_uid").size()
    uid_to_k = dict(zip(test_uid_lens.index, test_uid_lens.values))
    te_prefix = np.array([uid_to_k.get(u, 5) for u in test_uids_order])

    pred_before = ens_te_a.argmax(1).copy()
    masked = ens_te_a.copy()
    for i in range(len(masked)):
        k = int(te_prefix[i])
        blocked = mask_dict.get(k, set())
        if not blocked and k >= 2:
            blocked = {15, 16, 17, 18}     # 保險: stroke 2+ 永遠不會是發球
        for c in blocked:
            masked[i, c] = 0.0
        s = masked[i].sum()
        if s > 0:
            masked[i] /= s
        else:
            masked[i] = 1.0 / n_action
    n_changed = (pred_before != masked.argmax(1)).sum()
    print(f"  Position masking (min_count={min_count if False else 5}): "
          f"{n_changed} predictions changed ({n_changed/len(pred_before)*100:.1f}%)")
    return masked

## CELL 15b: Prior-Correction (per-class 機率乘子, 乾淨 +macroF1)


In [ ]:
# ============================================================
# CELL 15b: Prior-Correction (per-class probability multipliers)
# ============================================================
# 動機: macro-F1 對每個 class 等權平均, 模型對主類過度預測、對稀有類預測不足,
#   稀有類的低 recall 把 macro-F1 拉低。對 ensemble 機率乘上 per-class 乘子
#   (argmax(prob * mult)) 等同於在推論時重新平衡先驗, 可抬升弱類 F1。
# 作法: 在「乾淨 OOF」上用 coordinate ascent 直接最大化 macro-F1 求乘子,
#   再用 shrink 收縮回 1.0 防 overfit, 套用到 test 機率。
# 驗證: nested GroupKFold-by-match (在 train match 上調、held-out match 上評) 會 generalize,
#   故套到 test 合理。注意只動 action/point (非 server), 完全不涉及任何標籤洩漏。
# 已測: OOF action 0.3656->~0.373, point 0.2333->~0.242 (full-OOF tuned);
#   nested-CV 誠實增益 action +0.0017 / point +0.004。

PRIOR_CORR_GRID   = [0.3, 0.5, 0.7, 0.85, 1.0, 1.2, 1.5, 2.0, 3.0]
PRIOR_CORR_SHRINK = 0.0   # 已停用: OOF +0.0031 但 LB -0.0029 (0.3782->0.3753), 典型 OOF->LB gap; 設 0 即回乾淨 V5

def _macro_with_mult(labels, probs, mult):
    pred = (probs * mult).argmax(1)
    return f1_score(labels, pred, average="macro", zero_division=0)

def tune_prior_multipliers(labels, probs, n_class, grid=PRIOR_CORR_GRID, n_pass=8):
    """coordinate ascent: 逐類在 grid 上找能最大化 macro-F1 的乘子。"""
    mult = np.ones(n_class)
    for _ in range(n_pass):
        improved = False
        for c in range(n_class):
            best_s = _macro_with_mult(labels, probs, mult); best_g = mult[c]
            for g in grid:
                m2 = mult.copy(); m2[c] = g
                s = _macro_with_mult(labels, probs, m2)
                if s > best_s:
                    best_s, best_g, improved = s, g, True
            mult[c] = best_g
        if not improved:
            break
    return mult

def fit_prior_correction(oof_probs, oof_labels, oof_mask, n_class, label="",
                          shrink=PRIOR_CORR_SHRINK):
    """在乾淨 OOF 上求乘子並收縮; 回傳可直接乘到 test 機率的乘子向量。"""
    labels = oof_labels[oof_mask]; probs = oof_probs[oof_mask]
    base = _macro_with_mult(labels, probs, np.ones(n_class))
    mult = tune_prior_multipliers(labels, probs, n_class)
    mult_shrunk = np.exp(np.log(mult) * shrink)
    tuned = _macro_with_mult(labels, probs, mult_shrunk)
    print(f"  [{label}] prior-correction: OOF macro {base:.4f} -> {tuned:.4f} "
          f"(shrink={shrink}) | mult={np.round(mult_shrunk, 2).tolist()}")
    return mult_shrunk

def apply_prior_correction(test_probs, mult):
    """對 test 機率乘上乘子並 row-normalize (argmax 不變, 保持機率語意)。"""
    out = test_probs * mult[None, :]
    s = out.sum(1, keepdims=True)
    s[s == 0] = 1.0
    return out / s


## CELL 16: Main: 串接全流程 (NN → GBDT → ensemble → masking → 提交)

In [ ]:
def main():
    print("=" * 60); print("VV3: VV1's NN+GBDT + VV2A safe additions"); print("=" * 60)
    train_raw = pd.read_csv(CFG.train_path)
    test_raw  = pd.read_csv(CFG.test_path)

    old_uids = set()
    if CFG.old_test_path and os.path.exists(CFG.old_test_path):
        old_test = pd.read_csv(CFG.old_test_path)
        old_uids = set(old_test["rally_uid"])
        print(f"  train: {len(train_raw)} rows ({train_raw['match'].nunique()} matches) | "
              f"old test: {len(old_test)} rows ({old_test['match'].nunique()} matches)")
        combined_raw = pd.concat([train_raw, old_test], ignore_index=True)
        combined_raw = combined_raw.sort_values(["rally_uid", "strikeNumber"]).reset_index(drop=True)
        print(f"  combined: {len(combined_raw)} rows, {combined_raw['match'].nunique()} matches")
    else:
        combined_raw = train_raw

    print("\nComputing stats...")
    player_stats = compute_player_stats(combined_raw)
    sex_stats    = compute_sex_stats(combined_raw)
    cond_stats, act_stats, global_stats = compute_point_conditional_stats(combined_raw)
    bigram_stats, bigram_global = compute_action_bigram_stats(combined_raw)

    test_lens = test_raw.groupby("rally_uid").size().values
    test_dist = Counter(test_lens); total_te = sum(test_dist.values())

    train_df, encoders, player_encoder = preprocess(combined_raw, is_train=True,
                                                       player_stats=player_stats, sex_stats=sex_stats)
    test_df, _, _ = preprocess(test_raw, encoders=encoders, player_encoder=player_encoder,
                                player_stats=player_stats, sex_stats=sex_stats, is_train=False)
    train_df["is_old"] = train_df["rally_uid"].isin(old_uids).astype(int)
    print(f"  preprocessed train rows: {len(train_df)} rallies: {train_df['rally_uid'].nunique()}"
          f" | server-train rows(乾淨): {(train_df['is_old']==0).sum() if 'is_old' in train_df else len(train_df)} rallies")
    print(f"  preprocessed test rows:  {len(test_df)} rallies: {test_df['rally_uid'].nunique()}")

    # === NN training ===
    matches = train_df["match"].values
    fold_iter = list(GroupKFold(n_splits=CFG.n_folds).split(train_df, groups=matches))

    test_ds = TTDataset(test_df, mode="test")
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_uids_order = [s["uid"] for s in test_ds.samples]

    def run_nn_track(model_class, label):
        print("\n" + "=" * 60); print(f"NN training [{label}]"); print("=" * 60)
        print(f"  total models: {CFG.n_folds * CFG.n_seeds_nn}")
        net_oof = {}
        sum_a = {u: np.zeros(N_ACTION, dtype=np.float64) for u in test_uids_order}
        sum_p = {u: np.zeros(N_POINT,  dtype=np.float64) for u in test_uids_order}
        sum_s = {u: 0.0 for u in test_uids_order}
        n_models, fold_scores = 0, []
        for fold_idx, (tr_idx, va_idx) in enumerate(fold_iter):
            print("\n" + "-" * 50); print(f"[{label}] FOLD {fold_idx + 1}/{CFG.n_folds}"); print("-" * 50)
            tr_fold = train_df.iloc[tr_idx].reset_index(drop=True)
            va_fold = train_df.iloc[va_idx].reset_index(drop=True)
            val_loader_fold = DataLoader(TTDataset(va_fold, mode="train"), batch_size=CFG.batch_size,
                                           shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
            for seed_offset in range(CFG.n_seeds_nn):
                seed = CFG.seed + fold_idx * 100 + seed_offset * 13
                model, best = train_one_dl_fold(tr_fold, va_fold, fold_idx, seed, encoders,
                                                  player_encoder, test_dist, total_te, model_class=model_class)
                fold_scores.append(best)
                for uid, k, pa, pp, ps_b, ya, yp, yss in predict_with_keys(model, val_loader_fold):
                    key = (uid, k)
                    if key not in net_oof:
                        net_oof[key] = {"p_action": np.zeros(N_ACTION), "p_point": np.zeros(N_POINT),
                                        "p_server": 0.0, "n": 0, "y_action": ya, "y_point": yp, "y_server": yss}
                    net_oof[key]["p_action"] += pa; net_oof[key]["p_point"] += pp
                    net_oof[key]["p_server"] += ps_b; net_oof[key]["n"] += 1
                for uid, k, pa, pp, ps_b, _, _, _ in predict_with_keys(model, test_loader):
                    sum_a[uid] += pa; sum_p[uid] += pp; sum_s[uid] += ps_b
                n_models += 1; del model
                if device.type == "cuda": torch.cuda.empty_cache()
        for key in net_oof:
            net_oof[key]["p_action"] /= net_oof[key]["n"]
            net_oof[key]["p_point"]  /= net_oof[key]["n"]
            net_oof[key]["p_server"] /= net_oof[key]["n"]
        print(f"\n  [{label}] mean fold score: {np.mean(fold_scores):.4f} \u00b1 {np.std(fold_scores):.4f} ({n_models} models)")
        avg_a = np.array([sum_a[u] / n_models for u in test_uids_order])
        avg_p = np.array([sum_p[u] / n_models for u in test_uids_order])
        avg_s = np.array([sum_s[u] / n_models for u in test_uids_order])
        return net_oof, avg_a, avg_p, avg_s

    net_oof, avg_action_tr_te, avg_point_tr_te, avg_server_tr_te = run_nn_track(MultiTaskGRUV3, "GRU")
    if getattr(CFG, "use_transformer", False):
        tfm_oof, tfm_a_te, tfm_p_te, tfm_s_te = run_nn_track(MultiTaskTransformerV3, "Transformer")
    if getattr(CFG, "use_shuttlenet", False):
        shu_oof, shu_a_te, shu_p_te, shu_s_te = run_nn_track(MultiTaskShuttleNetV3, "ShuttleNet")
    if getattr(CFG, "use_lstm", False):
        lstm_oof, lstm_a_te, lstm_p_te, lstm_s_te = run_nn_track(MultiTaskLSTMV3, "LSTM")
    if getattr(CFG, "use_tcn", False):
        tcn_oof, tcn_a_te, tcn_p_te, tcn_s_te = run_nn_track(MultiTaskTCNV3, "TCN")

    # === GBDT training (multi-seed) ===
    gbdt = train_gbdt_track(combined_raw, test_raw, player_stats, sex_stats,
                              cond_stats, act_stats, global_stats,
                              bigram_stats, bigram_global,
                              test_dist, total_te, label="GBDT", old_uids=old_uids)
    gbdt_keys = gbdt["oof_uid_k"]

    # Align NN OOF to GBDT key order
    nn_oof_a = np.zeros((len(gbdt_keys), N_ACTION))
    nn_oof_p = np.zeros((len(gbdt_keys), N_POINT))
    nn_oof_s = np.zeros(len(gbdt_keys))
    for i, key in enumerate(gbdt_keys):
        if key in net_oof:
            nn_oof_a[i] = net_oof[key]["p_action"]
            nn_oof_p[i] = net_oof[key]["p_point"]
            nn_oof_s[i] = net_oof[key]["p_server"]

    # --- Transformer 混入 NN 端: 在 OOF 上搜 wT, 沒幫助自動 0, 零風險 ---
    if getattr(CFG, "use_transformer", False):
        t_oof_a = np.zeros((len(gbdt_keys), N_ACTION))
        t_oof_p = np.zeros((len(gbdt_keys), N_POINT))
        t_oof_s = np.zeros(len(gbdt_keys))
        for i, key in enumerate(gbdt_keys):
            if key in tfm_oof:
                t_oof_a[i] = tfm_oof[key]["p_action"]
                t_oof_p[i] = tfm_oof[key]["p_point"]
                t_oof_s[i] = tfm_oof[key]["p_server"]
        mfill = gbdt["oof_filled"]
        labA = gbdt["oof_label_action"][mfill]; labP = gbdt["oof_label_point"][mfill]
        labS = gbdt["oof_label_server"][mfill]
        grid = [round(0.05 * i, 2) for i in range(0, 11)]
        f1m = lambda lab, P: f1_score(lab, P[mfill].argmax(1), average="macro", zero_division=0)
        def auc_m(P):
            try: return roc_auc_score(labS, P[mfill])
            except: return 0.5
        wTa = max(grid, key=lambda w: f1m(labA, (1 - w) * nn_oof_a + w * t_oof_a))
        wTp = max(grid, key=lambda w: f1m(labP, (1 - w) * nn_oof_p + w * t_oof_p))
        wTs = max(grid, key=lambda w: auc_m((1 - w) * nn_oof_s + w * t_oof_s))
        print(f"  Transformer 混入權重 wT=(action={wTa}, point={wTp}, server={wTs})  [0=沒幫助, 自動忽略]")
        nn_oof_a = (1 - wTa) * nn_oof_a + wTa * t_oof_a
        nn_oof_p = (1 - wTp) * nn_oof_p + wTp * t_oof_p
        nn_oof_s = (1 - wTs) * nn_oof_s + wTs * t_oof_s
        avg_action_tr_te = (1 - wTa) * avg_action_tr_te + wTa * tfm_a_te
        avg_point_tr_te  = (1 - wTp) * avg_point_tr_te  + wTp * tfm_p_te
        avg_server_tr_te = (1 - wTs) * avg_server_tr_te + wTs * tfm_s_te

    if getattr(CFG, "use_shuttlenet", False):
        s_oof_a = np.zeros((len(gbdt_keys), N_ACTION)); s_oof_p = np.zeros((len(gbdt_keys), N_POINT))
        s_oof_s = np.zeros(len(gbdt_keys))
        for i, key in enumerate(gbdt_keys):
            if key in shu_oof:
                s_oof_a[i] = shu_oof[key]["p_action"]; s_oof_p[i] = shu_oof[key]["p_point"]
                s_oof_s[i] = shu_oof[key]["p_server"]
        mfill = gbdt["oof_filled"]
        labA = gbdt["oof_label_action"][mfill]; labP = gbdt["oof_label_point"][mfill]; labS = gbdt["oof_label_server"][mfill]
        grid = [round(0.05 * i, 2) for i in range(0, 11)]
        f1m = lambda lab, P: f1_score(lab, P[mfill].argmax(1), average="macro", zero_division=0)
        def auc_m(P):
            try: return roc_auc_score(labS, P[mfill])
            except: return 0.5
        wSa = max(grid, key=lambda w: f1m(labA, (1 - w) * nn_oof_a + w * s_oof_a))
        wSp = max(grid, key=lambda w: f1m(labP, (1 - w) * nn_oof_p + w * s_oof_p))
        wSs = max(grid, key=lambda w: auc_m((1 - w) * nn_oof_s + w * s_oof_s))
        print(f"  ShuttleNet 混入權重 wS=(action={wSa}, point={wSp}, server={wSs})  [0=沒幫助, 自動忽略]")
        nn_oof_a = (1 - wSa) * nn_oof_a + wSa * s_oof_a
        nn_oof_p = (1 - wSp) * nn_oof_p + wSp * s_oof_p
        nn_oof_s = (1 - wSs) * nn_oof_s + wSs * s_oof_s
        avg_action_tr_te = (1 - wSa) * avg_action_tr_te + wSa * shu_a_te
        avg_point_tr_te  = (1 - wSp) * avg_point_tr_te  + wSp * shu_p_te
        avg_server_tr_te = (1 - wSs) * avg_server_tr_te + wSs * shu_s_te

    if getattr(CFG, "use_lstm", False):
        x_oof_a = np.zeros((len(gbdt_keys), N_ACTION)); x_oof_p = np.zeros((len(gbdt_keys), N_POINT))
        x_oof_s = np.zeros(len(gbdt_keys))
        for i, key in enumerate(gbdt_keys):
            if key in lstm_oof:
                x_oof_a[i] = lstm_oof[key]["p_action"]; x_oof_p[i] = lstm_oof[key]["p_point"]
                x_oof_s[i] = lstm_oof[key]["p_server"]
        mfill = gbdt["oof_filled"]
        labA = gbdt["oof_label_action"][mfill]; labP = gbdt["oof_label_point"][mfill]; labS = gbdt["oof_label_server"][mfill]
        grid = [round(0.05 * i, 2) for i in range(0, 11)]
        f1m = lambda lab, P: f1_score(lab, P[mfill].argmax(1), average="macro", zero_division=0)
        def auc_m(P):
            try: return roc_auc_score(labS, P[mfill])
            except: return 0.5
        wXa = max(grid, key=lambda w: f1m(labA, (1 - w) * nn_oof_a + w * x_oof_a))
        wXp = max(grid, key=lambda w: f1m(labP, (1 - w) * nn_oof_p + w * x_oof_p))
        wXs = max(grid, key=lambda w: auc_m((1 - w) * nn_oof_s + w * x_oof_s))
        print(f"  LSTM 混入權重 wX=(action={wXa}, point={wXp}, server={wXs})  [0=沒幫助, 自動忽略]")
        nn_oof_a = (1 - wXa) * nn_oof_a + wXa * x_oof_a
        nn_oof_p = (1 - wXp) * nn_oof_p + wXp * x_oof_p
        nn_oof_s = (1 - wXs) * nn_oof_s + wXs * x_oof_s
        avg_action_tr_te = (1 - wXa) * avg_action_tr_te + wXa * lstm_a_te
        avg_point_tr_te  = (1 - wXp) * avg_point_tr_te  + wXp * lstm_p_te
        avg_server_tr_te = (1 - wXs) * avg_server_tr_te + wXs * lstm_s_te

    if getattr(CFG, "use_tcn", False):
        x_oof_a = np.zeros((len(gbdt_keys), N_ACTION)); x_oof_p = np.zeros((len(gbdt_keys), N_POINT))
        x_oof_s = np.zeros(len(gbdt_keys))
        for i, key in enumerate(gbdt_keys):
            if key in tcn_oof:
                x_oof_a[i] = tcn_oof[key]["p_action"]; x_oof_p[i] = tcn_oof[key]["p_point"]
                x_oof_s[i] = tcn_oof[key]["p_server"]
        mfill = gbdt["oof_filled"]
        labA = gbdt["oof_label_action"][mfill]; labP = gbdt["oof_label_point"][mfill]; labS = gbdt["oof_label_server"][mfill]
        grid = [round(0.05 * i, 2) for i in range(0, 11)]
        f1m = lambda lab, P: f1_score(lab, P[mfill].argmax(1), average="macro", zero_division=0)
        def auc_m(P):
            try: return roc_auc_score(labS, P[mfill])
            except: return 0.5
        wXa = max(grid, key=lambda w: f1m(labA, (1 - w) * nn_oof_a + w * x_oof_a))
        wXp = max(grid, key=lambda w: f1m(labP, (1 - w) * nn_oof_p + w * x_oof_p))
        wXs = max(grid, key=lambda w: auc_m((1 - w) * nn_oof_s + w * x_oof_s))
        print(f"  TCN 混入權重 wX=(action={wXa}, point={wXp}, server={wXs})  [0=沒幫助, 自動忽略]")
        nn_oof_a = (1 - wXa) * nn_oof_a + wXa * x_oof_a
        nn_oof_p = (1 - wXp) * nn_oof_p + wXp * x_oof_p
        nn_oof_s = (1 - wXs) * nn_oof_s + wXs * x_oof_s
        avg_action_tr_te = (1 - wXa) * avg_action_tr_te + wXa * tcn_a_te
        avg_point_tr_te  = (1 - wXp) * avg_point_tr_te  + wXp * tcn_p_te
        avg_server_tr_te = (1 - wXs) * avg_server_tr_te + wXs * tcn_s_te

    # === Weight search (VV3: expanded range) ===
    if CFG.do_weight_search:
        best_w, _, _ = search_optimal_weights(
            nn_oof_a, nn_oof_p, nn_oof_s,
            gbdt["oof_action"], gbdt["oof_point"], gbdt["oof_server"],
            gbdt["oof_label_action"], gbdt["oof_label_point"], gbdt["oof_label_server"],
            base_weights=(CFG.gbdt_action_weight, CFG.gbdt_point_weight, CFG.gbdt_server_weight),
            mask=gbdt["oof_filled"],
        )
        wa, wp, ws = best_w
    else:
        wa, wp, ws = CFG.gbdt_action_weight, CFG.gbdt_point_weight, CFG.gbdt_server_weight

    # === Final ensemble ===
    ens_oof_a = (1 - wa) * nn_oof_a + wa * gbdt["oof_action"]
    ens_oof_p = (1 - wp) * nn_oof_p + wp * gbdt["oof_point"]
    ens_oof_s = (1 - ws) * nn_oof_s + ws * gbdt["oof_server"]

    uid_to_idx_te = {u: i for i, u in enumerate(test_uids_order)}
    ens_te_a = np.zeros((len(test_uids_order), N_ACTION))
    ens_te_p = np.zeros((len(test_uids_order), N_POINT))
    ens_te_s = np.zeros(len(test_uids_order))
    for i, (uid, k) in enumerate(gbdt["te_uid_k"]):
        ti = uid_to_idx_te[uid]
        ens_te_a[ti] = (1 - wa) * avg_action_tr_te[ti] + wa * gbdt["te_action"][i]
        ens_te_p[ti] = (1 - wp) * avg_point_tr_te[ti]  + wp * gbdt["te_point"][i]
        ens_te_s[ti] = (1 - ws) * avg_server_tr_te[ti] + ws * gbdt["te_server"][i]

    mask = gbdt["oof_filled"]
    f1a_oof = f1_score(gbdt["oof_label_action"][mask], ens_oof_a[mask].argmax(1),
                        average="macro", zero_division=0)
    f1p_oof = f1_score(gbdt["oof_label_point"][mask], ens_oof_p[mask].argmax(1),
                        average="macro", zero_division=0)
    try: auc_oof = roc_auc_score(gbdt["oof_label_server"][mask], ens_oof_s[mask])
    except: auc_oof = 0.5
    score_oof = 0.4 * f1a_oof + 0.4 * f1p_oof + 0.2 * auc_oof
    print(f"\nFinal OOF (VV3 ensemble): {score_oof:.4f} "
          f"(f1a={f1a_oof:.4f}, f1p={f1p_oof:.4f}, auc={auc_oof:.4f}) | "
          f"weights=({wa:.2f}, {wp:.2f}, {ws:.2f})")
    print(f"  VV1 OOF was: 0.3589 (LB 0.3833)")
    print(f"  VV2A OOF was: 0.3541 (LB 0.3812)")
    print(f"  Δ vs VV1: {score_oof - 0.3589:+.4f}")

    # === CELL 15 套用: Position-Aware Class Masking (+0.013 LB) ===
    print("\nApplying position-aware class masking...")
    mask_dict = compute_position_mask(combined_raw, min_count=5)
    for k in sorted(mask_dict.keys())[:5]:
        print(f"  k={k}: block {sorted(mask_dict[k])}")
    ens_te_a = apply_action_mask(ens_te_a, test_uids_order, test_raw, mask_dict)

    # === CELL 15b 套用: Prior-Correction (per-class 乘子, 乾淨 +macroF1) ===
    if PRIOR_CORR_SHRINK > 0:
        print("\nApplying prior-correction (per-class multipliers)...")
        mfill = gbdt["oof_filled"]
        mult_a = fit_prior_correction(ens_oof_a, gbdt["oof_label_action"], mfill,
                                      N_ACTION, label="action")
        mult_p = fit_prior_correction(ens_oof_p, gbdt["oof_label_point"], mfill,
                                      N_POINT, label="point")
        ens_te_a = apply_prior_correction(ens_te_a, mult_a)
        ens_te_p = apply_prior_correction(ens_te_p, mult_p)

    pred_action = ens_te_a.argmax(1).astype(int)
    pred_point  = ens_te_p.argmax(1).astype(int)
    print(f"\nAction class hist: {dict(sorted(Counter(pred_action.tolist()).items()))}")
    print(f"Point  class hist: {dict(sorted(Counter(pred_point.tolist()).items()))}")

    # === Submission ===
    submission = pd.DataFrame({"rally_uid": test_uids_order, "actionId": pred_action,
                                "pointId": pred_point, "serverGetPoint": ens_te_s})
    try:
        sample = pd.read_csv(CFG.sample_path)
        if "rally_uid" in sample.columns and len(sample) > 0:
            submission = sample[["rally_uid"]].merge(submission, on="rally_uid", how="left")
            submission["actionId"]       = submission["actionId"].fillna(0).astype(int)
            submission["pointId"]        = submission["pointId"].fillna(0).astype(int)
            submission["serverGetPoint"] = submission["serverGetPoint"].fillna(0.5)
    except Exception as e:
        print("sample align skipped:", e)
    submission.to_csv(CFG.output_path, index=False)
    print(f"\n✓ Saved {CFG.output_path}, shape={submission.shape}")

    # === Save npz for diagnostics ===
    npz_path = CFG.output_path.replace(".csv", "_preds.npz")
    try:
        np.savez(npz_path,
            test_uids=np.array(test_uids_order),
            ens_test_action=ens_te_a, ens_test_point=ens_te_p, ens_test_server=ens_te_s,
            ens_oof_action=ens_oof_a, ens_oof_point=ens_oof_p, ens_oof_server=ens_oof_s,
            nn_oof_action=nn_oof_a, nn_oof_point=nn_oof_p, nn_oof_server=nn_oof_s,
            nn_test_action=avg_action_tr_te, nn_test_point=avg_point_tr_te, nn_test_server=avg_server_tr_te,
            gbdt_oof_action=gbdt["oof_action"], gbdt_oof_point=gbdt["oof_point"], gbdt_oof_server=gbdt["oof_server"],
            gbdt_test_action=gbdt["te_action"], gbdt_test_point=gbdt["te_point"], gbdt_test_server=gbdt["te_server"],
            oof_uid_k=np.array(gbdt_keys, dtype=object),
            oof_label_action=gbdt["oof_label_action"],
            oof_label_point=gbdt["oof_label_point"],
            oof_label_server=gbdt["oof_label_server"],
            oof_filled=gbdt["oof_filled"],
            best_weights=np.array([wa, wp, ws]),
        )
        print(f"✓ Saved npz")
    except Exception as e:
        print("npz save skipped:", e)

## CELL 17: 執行

In [ ]:
main()

VV3: VV1's NN+GBDT + VV2A safe additions
  train: 84707 rows (216 matches) | old test: 3589 rows (55 matches)
  combined: 88296 rows, 271 matches

Computing stats...
  conditional pt: 63 (pos, act) pairs
  action bigram → next-a: 123 (p2, p1) pairs with count >= 30
  preprocessed train rows: 88296 rallies: 16231
  preprocessed test rows:  5668 rallies: 1845

NN training (Bi-GRU + rally_continue)
  total NN models: 15

--------------------------------------------------
FOLD 1/5
--------------------------------------------------
  fold 0 seed 42 | tr 57699 va 14366
  Ep 01 | loss 2.1486 | score 0.2867 | f1a 0.2566 f1p 0.1661 auc 0.5883
  Ep 02 | loss 2.0267 | score 0.3078 | f1a 0.3049 f1p 0.1710 auc 0.5870
  Ep 03 | loss 1.9881 | score 0.3024 | f1a 0.2930 f1p 0.1700 auc 0.5858
  Ep 04 | loss 1.9526 | score 0.3015 | f1a 0.2859 f1p 0.1733 auc 0.5892
  Ep 05 | loss 1.9331 | score 0.3060 | f1a 0.2955 f1p 0.1752 auc 0.5888
  Ep 06 | loss 1.9076 | score 0.3106 | f1a 0.3025 f1p 0.1789 auc 0.590